# AI工学101 — 第28回

## モデルの中身を読む：係数・特徴量重要度・Permutation Importance

第27回では、**特徴量を選ぶ・減らす**ところまで来ました。

今日はその逆方向に一歩踏み込みます。

> **「このモデル、いったい何を見て予測してるの？」**

これを調べます。

機械学習では、Accuracyが高いだけでは不十分なことがあります。

例えば、

```text
モデルA
Accuracy = 97%
```

となっていても、

> 「なぜ97%なの？」

が分からないと、モデルを改善したり、異常な判断を発見したり、現場で信頼して使ったりするのが難しくなります。

そこで今日は、

```text
Logistic Regression
    ↓
coef_

Decision Tree / Random Forest
    ↓
feature_importances_

任意のモデル
    ↓
Permutation Importance
```

という3種類の「モデルを読む」方法を学びます。

---

# 🎯 今日のゴール

* `coef_` の意味を理解する
* 線形モデルの係数から予測への影響方向を読む
* `feature_importances_` の意味と限界を理解する
* Permutation Importanceを実装できる
* **「重要度」と「因果関係」は別物**だと理解する
* Pipeline越しに特徴量重要度を調べる方法を知る
* 複数の説明方法を比較できる

---

# 📖 講義：約20〜25分

## 1. 「説明可能」とは何か？

まず注意。

今日扱うのは、

> **モデルの内部や入力特徴量と予測の関係を調べる方法**

です。

これは必ずしも、

> 「AIが人間と同じ理由で判断していることを証明する」

ことではありません。

例えば、

```text
feature_importanceが高い
```

から、

```text
その特徴量が原因
```

とは言えません。

ここは第21回・第22回でも出てきた重要ポイントです。

---

# 2. Logistic Regressionの `coef_`

まず最も分かりやすいところから。

Logistic Regressionは概念的には、

$$
z = w_1x_1+w_2x_2+\cdots+w_dx_d+b
$$

という線形結合を作り、

$$
P(y=1)=\sigma(z)
$$

によって確率に変換します。

つまり、

```text
入力特徴量
 ↓
重み w
 ↓
合計
 ↓
確率
```

です。

だから、

```python
model.coef_
```

を見ると、

> **各特徴量にどんな重みが付いているか**

を見ることができます。

---

# 💻 実習1：Breast Cancerデータ

第24回・25回でも使ったデータを使います。

```python
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = data.data
y = data.target

feature_names = data.feature_names
```

分割。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

---

# 💻 実習2：StandardScaler + Logistic Regression

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

学習。

```python
model.fit(
    X_train,
    y_train
)
```

評価。

```python
print(
    model.score(
        X_test,
        y_test
    )
)
```

---

# 💻 実習3：係数を取り出す

Pipelineの中のLogistic Regressionを取り出します。

```python
classifier = model.named_steps[
    "classifier"
]
```

係数。

```python
coefficients = classifier.coef_[0]

print(
    coefficients
)
```

---

# 🧠 3. 係数の符号

例えば、

```text
feature A → +0.8
feature B → -0.5
feature C → +0.1
```

だったとします。

ざっくり、

```text
+0.8
```

なら、その特徴量が大きくなると、

**正例側のスコアを押し上げる方向**

に働きます。

```text
-0.5
```

なら、

**正例側のスコアを押し下げる方向**

です。

---

# ⚠️ ただし「係数の大きさ」はそのまま比較できない場合がある

これは非常に重要。

例えば、

```text
年齢
年収
```

では単位が違います。

```text
年齢 → 20〜80
年収 → 3,000,000〜10,000,000
```

このままだと係数の数字を単純比較しにくい。

だから今回、

```python
StandardScaler()
```

を使っています。

標準化された特徴量なら、

> **1標準偏差ぶん増えたときに、モデルの線形スコアがどちらへどの程度動くか**

という比較がしやすくなります。

---

# 💻 実習4：特徴量と係数を対応させる

```python
for name, coef in zip(
    feature_names,
    coefficients
):
    print(
        name,
        coef
    )
```

これだけでも、

```text
どの特徴量が
正例方向に働く？

どの特徴量が
負例方向に働く？
```

が見えてきます。

---

# 💻 実習5：絶対値で並べる

係数の符号ではなく、

```text
どれくらい大きいか
```

を見たい場合は絶対値を取ります。

```python
import numpy as np

importance = np.abs(
    coefficients
)
```

ランキング。

```python
indices = np.argsort(
    importance
)[::-1]
```

上位10個。

```python
for i in indices[:10]:

    print(
        feature_names[i],
        coefficients[i]
    )
```

---

# 🧠 4. ここで注意

係数が大きい

↓

**「その特徴量が重要」**

という解釈は、モデルの種類や前処理、特徴量の相関関係などを考慮する必要があります。

例えば、

```text
特徴量A
特徴量B
```

がほぼ同じ情報を持っている場合、

```text
A → +0.8
B → +0.1
```

となったからといって、

> 「Aだけが重要でBは重要ではない」

とは限りません。

**相関した特徴量に重みが分散する**ことがあります。

---

# 🌳 5. Decision Tree / Random Forestの重要度

次は木モデル。

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(
    X_train,
    y_train
)
```

重要度。

```python
importances = rf.feature_importances_
```

確認。

```python
for name, importance in zip(
    feature_names,
    importances
):

    print(
        name,
        importance
    )
```

---

# 🧠 6. `feature_importances_` は何を表している？

Random Forestでは、各決定木が、

```text
どの特徴量を使って
データを分割したか
```

をもとに、特徴量の重要度を計算できます。

つまり、

> **木の分割にどれくらい貢献したか**

を見る指標です。

---

# ⚠️ ただし、これにも弱点がある

特に、

```text
カテゴリ数が多い特徴量
```

や、

```text
連続値特徴量
```

などでは、単純なimpurity-based importanceが偏ることがあります。

また、

```text
AとBが強く相関
```

している場合、

```text
Aだけ高重要度
Bは低重要度
```

になることもあります。

そこで、

> **別の方法で重要度を測ってみよう**

となります。

---

# 🧪 7. Permutation Importance

今日の主役です。

考え方は非常にシンプル。

> **「その特徴量を壊してみたら、モデルの性能はどれくらい落ちる？」**

です。

例えば、

```text
元のデータ

age
income
height
weight
```

で、

```text
Accuracy = 95%
```

だったとします。

ここで、

```text
income
```

だけをランダムにシャッフルします。

すると、

```text
本来のincome
↓
他の人のincomeとランダムに入れ替わる
```

ので、

**incomeとターゲットの対応関係が壊れます。**

それで、

```text
Accuracy = 87%
```

になったなら、

```text
95% → 87%
```

なので、

**性能低下 = 8ポイント**

です。

この低下が大きいほど、

> **そのモデルにとって、その特徴量が重要だった**

と考えられます。

---

# 💻 実習6：Permutation Importance

```python
from sklearn.inspection import permutation_importance
```

実行。

```python
result = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="accuracy"
)
```

重要度。

```python
importance = result.importances_mean
```

標準偏差。

```python
std = result.importances_std
```

確認。

```python
for name, mean, sd in zip(
    feature_names,
    importance,
    std
):

    print(
        name,
        "mean=",
        mean,
        "std=",
        sd
    )
```

---

# 🧠 8. なぜPermutation Importanceが面白い？

モデルの種類に依存しにくいからです。

例えば、

```text
Logistic Regression
Random Forest
Gradient Boosting
```

のような異なるモデルでも、

> **「この特徴量を壊したら性能がどれだけ落ちる？」**

という同じ発想で調べられます。

つまり、

```text
モデル内部の仕組み
```

ではなく、

```text
入力特徴量を壊したときの予測性能
```

から重要度を測っています。

---

# 💻 実習7：Logistic RegressionにもPermutation Importance

```python
result_lr = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="accuracy"
)
```

平均。

```python
importance_lr = (
    result_lr.importances_mean
)
```

これで、

```text
Logistic Regression
```

についてもPermutation Importanceを計算できます。

---

# 🔥 ここで比較

同じデータに対して、

```text
Logistic Regression
    ↓
coef_

Random Forest
    ↓
feature_importances_

両方
    ↓
Permutation Importance
```

を比較できます。

これが今日のかなり重要な実験です。

---

# 🧠 9. 「重要度」は一種類ではない

例えば、

```text
feature A
```

について、

```text
Logistic coef → 大

Random Forest importance → 小

Permutation importance → 大
```

ということもあり得ます。

なぜでしょう？

**測っているものが違うからです。**

---

## `coef_`

```text
線形モデル内部の重み
```

---

## `feature_importances_`

```text
木の分割への貢献
```

---

## Permutation Importance

```text
その特徴量を壊したときの性能低下
```

---

# 🧠 10. そして「因果関係」ではない

今日の超重要ポイント。

例えば、

```text
靴の売上
```

と

```text
アイスクリームの売上
```

が強く相関していたとします。

モデルが、

```text
靴の売上
```

を重要特徴量として使っていたとしても、

> 「靴がアイスを売れる原因」

とは限りません。

例えば、

```text
夏
 ↓
アイス売上 ↑
靴売上 ↑
```

という共通原因があるかもしれません。

だから、

```text
予測上重要
```

と、

```text
因果的に重要
```

は別です。

これはデータサイエンスだけでなく、認知科学や実験研究でも非常に重要な区別です。

---

# 💻 実習8：重要度ランキングを作る

まずPermutation Importance。

```python
indices = np.argsort(
    result.importances_mean
)[::-1]
```

上位10個。

```python
for i in indices[:10]:

    print(
        feature_names[i],
        result.importances_mean[i]
    )
```

---

# 💻 実習9：可視化

```python
import matplotlib.pyplot as plt

top_n = 10

top_indices = indices[:top_n]

plt.barh(
    range(top_n),
    result.importances_mean[top_indices]
)

plt.yticks(
    range(top_n),
    feature_names[top_indices]
)

plt.xlabel(
    "Permutation Importance"
)

plt.gca().invert_yaxis()

plt.show()
```

これで、

```text
どの特徴量を壊すと
モデル性能が落ちるか
```

を視覚的に確認できます。

---

# 💻 実習10：重要度の「不確実性」も見る

Permutation Importanceでは、

```python
result.importances_std
```

もあります。

つまり、

> **その重要度がどれくらい安定しているか**

を見る手がかりになります。

例えば、

```text
Feature A
mean = 0.08
std  = 0.01
```

なら比較的安定。

一方、

```text
Feature B
mean = 0.06
std  = 0.05
```

なら、

> 「重要そうだけど、かなり揺れている」

という解釈になります。

---

# 🧪 実習11：Random ForestとGradient Boostingを比較

Gradient Boostingも作ります。

```python
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    random_state=42
)

gb.fit(
    X_train,
    y_train
)
```

Permutation Importance。

```python
result_gb = permutation_importance(
    gb,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="accuracy"
)
```

これで、

```text
Random Forest
vs
Gradient Boosting
```

の、

**「どの特徴量に依存しているか」**

まで比較できます。

---

# 💻 実習12：Cross Validationとの接続

ここは一段上の話。

1回のtest setだけで、

```text
Permutation Importance
```

を計算すると、そのデータ分割に依存します。

より慎重にやるなら、

```text
Fold 1
 ↓
Permutation Importance

Fold 2
 ↓
Permutation Importance

Fold 3
 ↓
...
```

として、

> **複数の分割で重要度が安定しているか**

を見ることができます。

今日は実装を簡単にするため、まずは通常のtest setで実験します。

ただし、

**「重要度にも不確実性がある」**

という視点を持っておいてください。

---

# ✍️ 演習

## 問1

Logistic Regressionについて、

```python
coef_
```

を取得してください。

その中から、

```text
絶対値が大きい特徴量
```

を5個確認します。

---

## 問2

Random Forestについて、

```python
feature_importances_
```

を取得してください。

上位5特徴量を確認します。

---

## 問3

Random Forestについて、

```python
permutation_importance()
```

を実行してください。

上位5特徴量を確認します。

---

## 問4

次の3つの違いを、自分の言葉で説明してください。

```text
coef_
feature_importances_
Permutation Importance
```

---

## 問5

もし、

```text
Feature A
```

がPermutation Importanceで高かったとしても、

> 「Feature Aが目的変数の原因だ」

とは言えない理由を説明してください。

---

# 👾 ボス戦：3種類の「重要度」を比較する

ここが今回の本丸です。

Breast Cancerデータについて、

```text
Logistic Regression
Random Forest
Gradient Boosting
```

の3モデルを作ります。

そして、

### Logistic Regression

```text
coef_
Permutation Importance
```

### Random Forest

```text
feature_importances_
Permutation Importance
```

### Gradient Boosting

```text
feature_importances_
Permutation Importance
```

を計算します。

そして、

> **各モデルが「重要だ」と判断している特徴量は一致しているか？**

を確認してください。

---

## ボス戦・考察

もし、

```text
Logistic Regression
→ Feature A

Random Forest
→ Feature B

Gradient Boosting
→ Feature C
```

のように結果が違ったら、

**どれが間違いなのでしょう？**

必ずしもそうではありません。

モデルが、

```text
異なる関数形
異なる仮定
異なる相互作用
```

を使っているため、

**データのどこを利用して予測しているかが変わる**

可能性があります。

ここまで考えられたら、第28回はかなり成功です。

---

# 🌱 今日のまとめ

今回の核心は、

> **「モデル性能」と「モデルが何を使っているか」は別の軸で評価する**

ことです。

```text
Accuracy = どれくらい当たる？

重要度 = 何を使っている？

```

そして重要度にも、

```text
coef_
    ↓
線形モデルの重み

feature_importances_
    ↓
木の分割への貢献

Permutation Importance
    ↓
特徴量を壊したときの性能低下
```

という違いがあります。

さらに、

```text
重要
≠
原因
```

です。

これは今日の最重要注意事項。

---

# 🧭 AI工学101・現在地

今のscikit-learn編は、かなり「AI開発の一連の流れ」になってきました。

```text
データ
 ↓
前処理
 ↓
特徴量
 ↓
特徴量選択 / PCA
 ↓
モデル
 ↓
学習
 ↓
予測確率
 ↓
Threshold
 ↓
評価
 ↓
Cross Validation
 ↓
Hyperparameter Search
 ↓
モデル解釈
 ↓
最終評価
```

つまり、

**「モデルを呼び出してAccuracyを見る」段階から、かなり離れてきています。**

---

# 🔜 第29回

## ハイパーパラメータ探索を本気でやる：GridSearchCV / RandomizedSearchCV

次回は第19回で触れたハイパーパラメータ探索を、もう一度**実務レベルで整理**します。

扱うのは、

* `GridSearchCV`
* `RandomizedSearchCV`
* パラメータ空間
* CVとtest setの役割
* 探索による過学習
* `best_params_`
* `best_score_`
* 最終test評価
* 探索コストと性能のトレードオフ

です。

ここで、

```text
「とりあえずGridSearchを回す」
```

から、

```text
「何を、なぜ、どの範囲で探索するのか」
```

という**実験設計**へ進みます。
